# Modeling — Prediksi Kunjungan Klinik Harian (Horizon 7 Hari)

**Sumber data:** `data_harian_kunjungan_2025.csv` (output notebook `cleaning_eda_kunjungan.ipynb`)

**Tujuan:** membandingkan 6 algoritma prediksi kunjungan harian per layanan, dengan cara prediksi **7 hari sekaligus** (fit per layanan → 6 × 8 = 48 model):
1. **Naive (mean)** — rata-rata data train
2. **Seasonal Naive (lag-7)** — nilai minggu lalu pada hari yang sama
3. **SARIMA** — grid search ringan via AIC, parameter dibekukan utk semua window
4. **Linear Regression** — fitur: one-hot hari, sin/cos bulan, trend
5. **Ridge** — regresi linear + regularisasi l2 (fitur & skema sama dengan LinReg)
6. **XGBoost Regressor** — gradient boosted trees dengan fitur yang sama

**Paradigma evaluasi:** rolling-origin 5 window (tiap window: train ditambah 7 hari, predik 7 hari ke depan) — hold-out total 35 hari terakhir.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 100

NAMA_HARI = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']

## MLflow Tracking

Semua hasil training dilog ke MLflow (backend SQLite, file `mlflow.db` di folder ini):

- **48 parent run** (6 model × 8 layanan) + **child run tuning**: Ridge (3 alpha) dan XGBoost (12 combo grid).
- Best tuning dipilih dari **MAE rata-rata 5 window rolling-origin**. Child run disarangkan di parent run (*nested runs*).
- Tiap run menyimpan **params** (hyperparameter), **metrics** (MAE & RMSE per window + rata-rata), **tags**, dan **artifacts**: model `.pkl` hasil fit tiap window, `prediksi.csv`, dan plot aktual vs prediksi.
- Rekap di akhir notebook + dashboard via `mlflow ui --backend-store-uri sqlite:///mlflow.db`. 

In [ ]:
import json
import os
import pickle
import tempfile

import mlflow

mlflow.set_tracking_uri('sqlite:///mlflow.db')
EXPERIMENT = 'prediksi-kunjungan-klinik'
mlflow.set_experiment(EXPERIMENT)

# Grid tuning (dipakai di sel Ridge & XGBoost)
ALPHAS_RIDGE = [0.1, 1.0, 10.0]
GRID_XGB = [{'n_estimators': ne, 'max_depth': md, 'learning_rate': lr}
            for ne in (100, 200, 400)
            for md in (2, 3)
            for lr in (0.01, 0.1)]

print('MLflow', mlflow.__version__, '| experiment:', EXPERIMENT)
print('Grid tuning:', len(ALPHAS_RIDGE), 'alpha Ridge /', len(GRID_XGB), 'combo XGBoost')

In [ ]:
def hitung_metrik_run(pred_run):
    metrik = {}
    for w, sub in pred_run.groupby('window'):
        mae = float(np.mean(np.abs(sub.y_aktual - sub.y_prediksi)))
        rmse = float(np.sqrt(np.mean((sub.y_aktual - sub.y_prediksi) ** 2)))
        metrik[f'mae_w{w}'] = round(mae, 4)
        metrik[f'rmse_w{w}'] = round(rmse, 4)
    metrik['mae_avg'] = round(float(np.mean([v for k, v in metrik.items() if k.startswith('mae_')])), 4)
    metrik['rmse_avg'] = round(float(np.mean([v for k, v in metrik.items() if k.startswith('rmse_')])), 4)
    return metrik


def log_run(nama_model, lay, params, pred_run, models_by_window, tipe='baseline', nested=False):
    "Logging 1 run MLflow: params + metrics per window + artifacts (pkl, csv, plot)."
    metrik = hitung_metrik_run(pred_run)
    with mlflow.start_run(run_name=f'{nama_model}-{lay}', nested=nested):
        mlflow.log_params({'model': nama_model, 'layanan': lay,
                           'horizon': WINDOW, 'n_window': len(windows), **(params or {})})
        mlflow.log_metrics(metrik)
        mlflow.set_tags({'tipe': tipe, 'peran': 'child' if nested else 'run'})
        with tempfile.TemporaryDirectory() as tmp:
            for w, obj in models_by_window.items():
                with open(os.path.join(tmp, f'win{w}.pkl'), 'wb') as f:
                    pickle.dump(obj, f)
            pred_run.to_csv(os.path.join(tmp, 'prediksi.csv'), index=False)
            fig, ax = plt.subplots(figsize=(7, 3.2))
            sub = (pred_run.groupby('tanggal')
                   .agg(y_aktual=('y_aktual', 'first'), y_prediksi=('y_prediksi', 'mean')))
            ax.plot(sub.index, sub.y_aktual, 'k.', ms=9, label='aktual')
            ax.plot(sub.index, sub.y_prediksi, 'o-', ms=3, label='prediksi')
            ax.set_title(f'{nama_model} - {lay} (35 hari hold-out)')
            ax.set_ylabel('kunjungan/hari')
            ax.tick_params(axis='x', rotation=30)
            ax.legend(fontsize=8)
            fig.tight_layout()
            fig.savefig(os.path.join(tmp, 'plot_aktual_vs_pred.png'), dpi=110)
            plt.close(fig)
            mlflow.log_artifacts(tmp)
    return metrik['mae_avg'], metrik['rmse_avg']


print('Helper MLflow siap.')

## 1. Load Data

In [ ]:
daily = pd.read_csv('Dataset/data_harian_kunjungan_2025.csv', parse_dates=['tanggal', 'minggu_start'])
print('Dimensi:', daily.shape)
print('Rentang :', daily.tanggal.min().date(), 'sampai', daily.tanggal.max().date())
daily.head()

## 2. Rencana Rolling-Origin: 5 Window

Data terakhir **35 hari (5 × 7 hari)** dijadikan zona uji. Model dilatih hanya dengan data SEBELUM zona uji masing-masing window, lalu mempredik 7 hari habis-habisan:

In [ ]:
tanggal_all = pd.DatetimeIndex(sorted(daily.tanggal.unique()))
WINDOW = 7
N_TEST = 5 * WINDOW

win_rows = []
for k in range(5):
    uji_mulai = tanggal_all[-(N_TEST) + k * WINDOW]
    uji_akhir = uji_mulai + pd.Timedelta(days=WINDOW - 1)
    train_akhir = uji_mulai - pd.Timedelta(days=1)
    win_rows.append({'window': k + 1,
                     'train_sampai': train_akhir.date(),
                     'uji_mulai': uji_mulai.date(),
                     'uji_akhir': uji_akhir.date(),
                     'n_hari_train': (uji_mulai - tanggal_all[0]).days})
windows = pd.DataFrame(win_rows)
windows

## 3. Matriks Target: Tanggal × Layanan

In [ ]:
ORDER_LAYANAN = (daily.groupby('layanan').kunjungan.sum()
                 .sort_values(ascending=False).index.tolist())
print('Urutan layanan:', ORDER_LAYANAN)

Y = daily.pivot(index='tanggal', columns='layanan', values='kunjungan').sort_index()
Y = Y[ORDER_LAYANAN]
print('Matriks target:', Y.shape)
Y.head()

## 4. Kerangka Prediksi

Semua model mengikuti kontrak yang sama: `predict(train, uji_dates, layanan) -> array 7 nilai`

In [ ]:
def get_train(lay, train_akhir):
    return Y.loc[:train_akhir, lay]

def get_uji(uji_mulai, uji_akhir, lay):
    return Y.loc[uji_mulai:uji_akhir, lay]

MODELS = ['Naive (mean)', 'Seasonal Naive (lag-7)', 'SARIMA', 'Linear Regression', 'Ridge', 'XGBoost']
RUNTIME = {m: 0.0 for m in MODELS}
records = []

## 5. Baseline

### 5.1 Naive (mean) — rata-rata train

In [ ]:
t0 = time.perf_counter()
for lay in ORDER_LAYANAN:
    pred_run = []
    models_by_window = {}
    for _, w in windows.iterrows():
        train_akhir = pd.Timestamp(w.train_sampai)
        uji = pd.date_range(w.uji_mulai, w.uji_akhir)
        mu = get_train(lay, train_akhir).mean()
        akt = get_uji(w.uji_mulai, w.uji_akhir, lay)
        for d, a in zip(uji, akt):
            pred_run.append({'window': w.window, 'tanggal': d, 'y_aktual': a, 'y_prediksi': float(mu)})
        models_by_window[w.window] = {'tipe': 'naive_mean', 'nilai_mean': float(mu)}
    pred_run = pd.DataFrame(pred_run)
    records.extend([{**r, 'layanan': lay, 'model': 'Naive (mean)'} for r in pred_run.to_dict('records')])
    log_run('Naive (mean)', lay, {'baseline': 'mean train'}, pred_run, models_by_window)
RUNTIME['Naive (mean)'] = time.perf_counter() - t0
print('Naive selesai, baris records tercatat:', len(records))

### 5.2 Seasonal Naive (lag-7) — nilai minggu lalu

In [ ]:
t0 = time.perf_counter()
for lay in ORDER_LAYANAN:
    pred_run = []
    models_by_window = {}
    for _, w in windows.iterrows():
        uji = pd.date_range(w.uji_mulai, w.uji_akhir)
        akt = get_uji(w.uji_mulai, w.uji_akhir, lay)
        pred = Y.loc[uji - pd.Timedelta(days=7), lay].to_numpy()
        for d, a, p in zip(uji, akt, pred):
            pred_run.append({'window': w.window, 'tanggal': d, 'y_aktual': a, 'y_prediksi': float(p)})
        models_by_window[w.window] = {'tipe': 'seasonal_naive', 'lag': 7}
    pred_run = pd.DataFrame(pred_run)
    records.extend([{**r, 'layanan': lay, 'model': 'Seasonal Naive (lag-7)'} for r in pred_run.to_dict('records')])
    log_run('Seasonal Naive (lag-7)', lay, {'baseline': 'lag-7'}, pred_run, models_by_window)
RUNTIME['Seasonal Naive (lag-7)'] = time.perf_counter() - t0
print('Seasonal Naive selesai, baris records:', len(records))

## 6. SARIMA — pilih order via AIC (window 1), lalu dibekukan

Grid kecil utk data 330 hari; seasonal weekly (1,0,0,7). Orde terbaik per layanan dipakai ulang utk semua window (realistis: di praktik orde biasanya conservative & freeze).

In [ ]:
GRID_ORDERS = [(1, 0, 0), (1, 0, 1), (2, 0, 1), (1, 1, 1), (0, 1, 1)]
SEASONAL = (1, 0, 0, 7)

best_orders = {}
AIC_TABLE = {}
t0 = time.perf_counter()
for lay in ORDER_LAYANAN:
    train = get_train(lay, windows.loc[windows.window == 1, 'train_sampai'].iloc[0])
    hasil = []
    for order in GRID_ORDERS:
        try:
            res = SARIMAX(train.astype(float), order=order, seasonal_order=SEASONAL,
                          enforce_stationarity=False, enforce_invertibility=False
                          ).fit(disp=False)
            hasil.append((order, round(res.aic, 1)))
        except Exception:
            hasil.append((order, np.inf))
    best_orders[lay] = min(hasil, key=lambda x: x[1])[0]
    AIC_TABLE[lay] = {str(o): (None if not np.isfinite(v) else float(v)) for o, v in hasil}
RUNTIME['SARIMA'] = time.perf_counter() - t0

pd.DataFrame({'layanan': best_orders.keys(),
              'order_terpilih': [f'{o}' for o in best_orders.values()]}).set_index('layanan')

In [ ]:
t0 = time.perf_counter()
for lay in ORDER_LAYANAN:
    pred_run = []
    models_by_window = {}
    for _, w in windows.iterrows():
        train_akhir = pd.Timestamp(w.train_sampai)
        uji = pd.date_range(w.uji_mulai, w.uji_akhir)
        train = get_train(lay, train_akhir).astype(float)
        try:
            res = SARIMAX(train, order=best_orders[lay], seasonal_order=SEASONAL,
                          enforce_stationarity=False, enforce_invertibility=False
                          ).fit(disp=False)
            pred = res.get_forecast(WINDOW).predicted_mean.to_numpy()
            model_fit = res
        except Exception:
            pred = np.full(WINDOW, train.mean())
            model_fit = {'tipe': 'fallback_mean', 'nilai_mean': float(train.mean())}
        akt = get_uji(w.uji_mulai, w.uji_akhir, lay)
        for d, a, p in zip(uji, akt, pred):
            pred_run.append({'window': w.window, 'tanggal': d, 'y_aktual': a, 'y_prediksi': float(p)})
        models_by_window[w.window] = model_fit
    pred_run = pd.DataFrame(pred_run)
    records.extend([{**r, 'layanan': lay, 'model': 'SARIMA'} for r in pred_run.to_dict('records')])
    log_run('SARIMA', lay,
            {'best_order': str(best_orders[lay]), 'seasonal_order': str(SEASONAL),
             'aic_grid': json.dumps(AIC_TABLE[lay])},
            pred_run, models_by_window, tipe='statistical')
RUNTIME['SARIMA'] += time.perf_counter() - t0
print('SARIMA rolling selesai, baris records:', len(records))

## 7. Linear Regression (per layanan)

Fitur: **one-hot dow** (drop 1 ref), **sin/cos bulan**, dan **trend** linear (hari ke-n sejak 2025-01-01).

In [ ]:
T0 = tanggal_all[0]


def fitur(dates, t0_ref):
    df = pd.DataFrame(index=dates)
    df['trend'] = (df.index - t0_ref).days
    for d_ in range(6):
        df[f'dow_{d_}'] = (df.index.dayofweek == d_).astype(int)
    df['sin_bulan'] = np.sin(2 * np.pi * df.index.month / 12)
    df['cos_bulan'] = np.cos(2 * np.pi * df.index.month / 12)
    return df


t0 = time.perf_counter()
for lay in ORDER_LAYANAN:
    pred_run = []
    models_by_window = {}
    for _, w in windows.iterrows():
        train_akhir = pd.Timestamp(w.train_sampai)
        uji = pd.date_range(w.uji_mulai, w.uji_akhir)
        Xtr = fitur(Y.loc[:train_akhir].index, T0)
        Xte = fitur(uji, T0)
        ytr = get_train(lay, train_akhir).astype(float).values
        reg = LinearRegression().fit(Xtr, ytr)
        pred = reg.predict(Xte)
        akt = get_uji(w.uji_mulai, w.uji_akhir, lay)
        for d, a, p in zip(uji, akt, pred):
            pred_run.append({'window': w.window, 'tanggal': d, 'y_aktual': a, 'y_prediksi': float(p)})
        models_by_window[w.window] = reg
    pred_run = pd.DataFrame(pred_run)
    records.extend([{**r, 'layanan': lay, 'model': 'Linear Regression'} for r in pred_run.to_dict('records')])
    log_run('Linear Regression', lay, {'fitur': 'one-hot dow + sin/cos bulan + trend'},
            pred_run, models_by_window, tipe='ml')
RUNTIME['Linear Regression'] = time.perf_counter() - t0
print('LinReg selesai, baris records:', len(records))

## 8. Ridge — regresi linear + regularisasi l2

Fitur yang sama dengan LinReg, tapi di-scale dulu (StandardScaler) karena Ridge sensitif terhadap skala. `alpha=1.0`.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

t0 = time.perf_counter()
best_alphas = {}
for lay in ORDER_LAYANAN:
    hasil_alpha = {}
    prediksi_by_combo = {}
    with mlflow.start_run(run_name=f'Ridge-{lay}'):
        mlflow.log_params({'model': 'Ridge', 'layanan': lay,
                           'gridsearch': str(ALPHAS_RIDGE), 'kriteria': 'mae_avg 5 window'})
        mlflow.set_tags({'tipe': 'tuning', 'peran': 'parent'})
        for alpha in ALPHAS_RIDGE:
            pred_run = []
            models_by_window = {}
            for _, w in windows.iterrows():
                train_akhir = pd.Timestamp(w.train_sampai)
                uji = pd.date_range(w.uji_mulai, w.uji_akhir)
                Xtr = fitur(Y.loc[:train_akhir].index, T0)
                Xte = fitur(uji, T0)
                ytr = get_train(lay, train_akhir).astype(float).values
                reg = make_pipeline(StandardScaler(), Ridge(alpha=alpha)).fit(Xtr, ytr)
                pred = reg.predict(Xte)
                akt = get_uji(w.uji_mulai, w.uji_akhir, lay)
                for d, a, p in zip(uji, akt, pred):
                    pred_run.append({'window': w.window, 'tanggal': d, 'y_aktual': a, 'y_prediksi': float(p)})
                models_by_window[w.window] = reg
            pred_run = pd.DataFrame(pred_run)
            mae_avg, rmse_avg = log_run(f'Ridge alpha={alpha}', lay, {'alpha': alpha},
                                        pred_run, models_by_window, tipe='tuning', nested=True)
            key = f'alpha={alpha}'
            hasil_alpha[key] = mae_avg
            prediksi_by_combo[key] = pred_run
        best_key = min(hasil_alpha, key=hasil_alpha.get)
        best_alphas[lay] = float(best_key.split('=')[1])
        mlflow.log_params({'best_alpha': best_alphas[lay]})
        mlflow.log_metrics({'mae_avg_best': hasil_alpha[best_key]})
        records.extend([{**r, 'layanan': lay, 'model': 'Ridge'}
                        for r in prediksi_by_combo[best_key].to_dict('records')])
RUNTIME['Ridge'] = time.perf_counter() - t0
print('Ridge gridsearch selesai, baris records:', len(records), '| best alpha:', best_alphas)

## 9. XGBoost Regressor — gradient boosted trees

Fitur sama; tree menangkap interaksi non-linear tanpa asumsi linear. Parameter konservatif (depth 3, 200 pohon, lr 0,05) untuk data kecil.

In [ ]:
from xgboost import XGBRegressor

t0 = time.perf_counter()
best_xgb = {}
for lay in ORDER_LAYANAN:
    hasil_combo = {}
    prediksi_by_combo = {}
    with mlflow.start_run(run_name=f'XGBoost-{lay}'):
        mlflow.log_params({'model': 'XGBoost', 'layanan': lay,
                           'gridsearch': str(GRID_XGB), 'kriteria': 'mae_avg 5 window',
                           'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42})
        mlflow.set_tags({'tipe': 'tuning', 'peran': 'parent'})
        for combo in GRID_XGB:
            pred_run = []
            models_by_window = {}
            for _, w in windows.iterrows():
                train_akhir = pd.Timestamp(w.train_sampai)
                uji = pd.date_range(w.uji_mulai, w.uji_akhir)
                Xtr = fitur(Y.loc[:train_akhir].index, T0)
                Xte = fitur(uji, T0)
                ytr = get_train(lay, train_akhir).astype(float).values
                xgbr = XGBRegressor(subsample=0.8, colsample_bytree=0.8,
                                    random_state=42, verbosity=0, **combo)
                xgbr.fit(Xtr, ytr)
                pred = xgbr.predict(Xte)
                akt = get_uji(w.uji_mulai, w.uji_akhir, lay)
                for d, a, p in zip(uji, akt, pred):
                    pred_run.append({'window': w.window, 'tanggal': d, 'y_aktual': a, 'y_prediksi': float(p)})
                models_by_window[w.window] = xgbr
            pred_run = pd.DataFrame(pred_run)
            mae_avg, rmse_avg = log_run(
                f'XGBoost ne={combo["n_estimators"]} md={combo["max_depth"]} lr={combo["learning_rate"]}',
                lay, {**combo, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42},
                pred_run, models_by_window, tipe='tuning', nested=True)
            key = json.dumps(combo)
            hasil_combo[key] = mae_avg
            prediksi_by_combo[key] = pred_run
        best_key = min(hasil_combo, key=hasil_combo.get)
        best_xgb[lay] = json.loads(best_key)
        mlflow.log_params({'best_n_estimators': best_xgb[lay]['n_estimators'],
                           'best_max_depth': best_xgb[lay]['max_depth'],
                           'best_learning_rate': best_xgb[lay]['learning_rate']})
        mlflow.log_metrics({'mae_avg_best': hasil_combo[best_key]})
        records.extend([{**r, 'layanan': lay, 'model': 'XGBoost'}
                        for r in prediksi_by_combo[best_key].to_dict('records')])
RUNTIME['XGBoost'] = time.perf_counter() - t0
print('XGBoost gridsearch selesai, baris records:', len(records), '| best combo:', best_xgb)

## 10. Tabel Prediksi & Metrik

In [ ]:
pred_long = pd.DataFrame(records)
pred_long.y_prediksi = pred_long.y_prediksi.clip(lower=0)
assert len(pred_long) == 5 * 8 * 6 * 7
print('Total baris pred:', len(pred_long))
pred_long.sort_values(['layanan', 'window', 'tanggal', 'model']).head(12)

In [ ]:
def agg_metrik(df_):
    return pd.Series({
        'mae': np.mean(np.abs(df_.y_aktual - df_.y_prediksi)),
        'rmse': np.sqrt(np.mean((df_.y_aktual - df_.y_prediksi) ** 2)),
        'mape_kecil': np.nan,
    })

metrik = pred_long.groupby(['layanan', 'model']).apply(agg_metrik).drop(columns='mape_kecil')
metrik.head(8)

In [ ]:
mae_w = metrik['mae'].unstack('model').reindex(ORDER_LAYANAN)[MODELS]
rmse_w = metrik['rmse'].unstack('model').reindex(ORDER_LAYANAN)[MODELS]

print('=== MAE per Layanan × Model ===')
print(mae_w.round(3))
print()
print('=== RMSE per Layanan × Model ===')
print(rmse_w.round(3))

In [ ]:
agg = mae_w.mean(axis=0).rename('MAE (rata-rata)').to_frame()
agg['RMSE (rata-rata)'] = rmse_w.mean(axis=0)
agg['runtime_total'] = pd.Series(RUNTIME).reindex(agg.index).round(2)
agg['waktu_str'] = agg.runtime_total.apply(lambda s: f'{s:.1f}s')
agg.round(3)

In [ ]:
pemenang = mae_w.idxmin(axis=1).to_frame('model_terbaik')
pemenang['mae'] = mae_w.min(axis=1).round(2)
pemenang['jumlah_menang_modelnya'] = pemenang.model_terbaik.map(pemenang.model_terbaik.value_counts())
pemenang

## 13. Kesimpulan Eksperimen (6 model)

**Tabel agregat (5 window × 8 layanan):**

| Model | MAE | RMSE | Runtime |
|---|---|---|---|
| Naive (mean) | **0,440** | **0,577** | 0,1 s |
| SARIMA | 0,442 | 0,583 | 76,9 s |
| Linear Regression | 0,444 | 0,600 | 1,0 s |
| Ridge | 0,444 | 0,599 | 1,2 s |
| XGBoost | 0,484 | 0,635 | 67,1 s |
| Seasonal Naive (lag-7) | 0,493 | 0,767 | 0,1 s |

**Temuan:**

1. **Empat model teratas praktis seri** (Naive, SARIMA, LinReg, Ridge; selisih MAE < 0,004) — tak ada yang unggul signifikan di atas baseline rata-rata. Konsisten dengan EDA: autokorelasi lemah, pola hari tidak signifikan → series harian memang minim sinyal.
2. **Ridge ≡ LinReg** (beda < 0,001) — pada 9 fitur dan 330 sampel, regularisasi l2 tidak menghasilkan koefisien berbeda. Jadi untuk project ini cukup pilih salah satu.
3. **XGBoost lebih buruk dari baseline** (MAE 0,484 vs 0,440) dan runtime mahal (67 s) — boosted trees cenderung overfit noise pada series kecil yang hampir i.i.d.; hanya menang 1 layanan (Imunisasi, dengan margin tipis 0,630 vs 0,635).
4. **Seasonal Naive tetap terburuk** (RMSE 0,767) — menyalin minggu lalu menambah error karena lag-7 lemah.
5. **Pemenang per layanan tersebar**: LinReg menang 3 (USG, Pijat bayi, Program kehamilan — layanan kecil), Naive 2 (KB, Ibu hamil), SARIMA 1 (Berobat, layanan trend naik), XGBoost 1 (Imunisasi), Seasonal Naive 1 (Ibu bersalin).
6. **Biaya**: SARIMA 770× lebih mahal dari Naive dengan akurasi setara → tidak layak sebagai model produksi.

**Rekomendasi model akhir:**
- Produksi harian 7-ke-depan per layanan: **Naive (mean)** (paling murah, MAE terbaik), atau **LinReg/Ridge** jika butuh interpretabilitas trend/season for reporting.
- Layanan kecil (USG, Pijat bayi, Program kehamilan) bisa pakai LinReg karena sistematis menang di sana.
- Untuk perbaikan selanjutnya: feature eksternal (insentif/non-insentif hari), agregasi mingguan yang lebih stabil, atau pendekatan Poisson per layanan.


## 11. Visualisasi Perbandingan

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
mae_w.plot(kind='bar', ax=ax, width=0.78)
ax.set_title('MAE per Layanan (5 window × 7 hari)')
ax.set_ylabel('MAE kunjungan/hari')
ax.set_xlabel('layanan')
ax.tick_params(axis='x', rotation=30)
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, lay in zip(axes, ['KB', 'Berobat']):
    sub = pred_long[(pred_long.layanan == lay) & (pred_long.window == 5)]
    sub = sub.pivot(index='tanggal', columns='model', values='y_prediksi')
    akt = pred_long[(pred_long.layanan == lay) & (pred_long.window == 5)] \
        .groupby('tanggal').y_aktual.first()
    ax.plot(akt.index, akt.values, 'k.', ms=12, mec='black', label='Aktual')
    for m in MODELS:
        ax.plot(akt.index, sub[m], marker='o', lw=1.4, label=m)
    ax.set_title(f'Prediksi 7 hari terakhir — {lay}')
    ax.set_ylabel('kunjungan/hari')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 12. Ekspor Hasil

In [ ]:
metrik.reset_index().to_csv('evaluasi_model_per_layanan.csv', index=False)
pred_long.to_csv('prediksi_semua_model.csv', index=False)
print('evaluasi_model_per_layanan.csv :', metrik.shape)
print('prediksi_semua_model.csv       :', pred_long.shape)

## 14. Rekap Run MLflow

48 parent run (per model × layanan, termasuk parent tuning Ridge & XGBoost). Child run tuning (3 alpha Ridge + 12 combo XGBoost per layanan) bisa dilihat di dashboard MLflow UI.

In [ ]:
exp = mlflow.get_experiment_by_name(EXPERIMENT)
runs = mlflow.search_runs([exp.experiment_id], order_by=['metrics.mae_avg ASC'])
print('Total run MLflow tercatat:', len(runs))

tampil = runs[runs['tags.peran'].isin(['run', 'parent'])][
    ['tags.mlflow.runName', 'tags.tipe', 'metrics.mae_avg', 'metrics.rmse_avg', 'status']].copy()
tampil.columns = ['run', 'tipe', 'mae_avg', 'rmse_avg', 'status']
print('\nDashboard UI: `mlflow ui --backend-store-uri sqlite:///mlflow.db` → http://localhost:5000')
tampil.round(3)